In [ ]:
from anthropic import Anthropic

client = Anthropic()

response = client.messages.create(
    model="claude-sonnet-4-6",
    max_tokens=1024,
    messages=[
        {"role": "user", "content": "What is the capital of Bulgaria?"}
    ]
)

response.content[0].text 

In [ ]:
from litellm import completion
# First call
response1 = completion(
    model="claude-sonnet-4-6",
    messages=[{"role": "user", "content": "My name is Ivona."}]
)
print(response1.choices[0].message.content)

# Second call
response2 = completion(
    model="claude-opus-5-5",
    messages=[{"role": "user", "content": "What is my name?"}]
)
print(response2.choices[0].message.content)

In [ ]:
#Maintaining conversation history
messages = []

# First exchange
messages.append({"role": "user", "content": "My name is Ivona."})
response1 = completion(model="claude-sonnet-4-6", messages=messages)
assistant_message1 = response1.choices[0].message.content
messages.append({"role": "assistant", "content": assistant_message1})
print(assistant_message1)

# Second exchange - includes previous conversation history
messages.append({"role": "user", "content": "What is my name?"})
response2 = completion(model="claude-sonnet-4-6", messages=messages)
assistant_message2 = response2.choices[0].message.content
print(assistant_message2)

In [ ]:
from pydantic import BaseModel

"""Ssmple structured output using Pydantic models and response_format parameter"""
class ExtractedInfo(BaseModel):
    name: str
    email: str
    phone: str | None = None

response = completion(
    model="claude-sonnet-4-6",
    messages=[{
        "role": "user",
        "content": "My name is Shon, my email is shon@example.com, and my phone is 555-1234."
    }],
    response_format=ExtractedInfo
)

result = response.choices[0].message.content
print(result)
info = ExtractedInfo.model_validate_json(result)

In [ ]:
import asyncio
from litellm import acompletion

"""Example of making multiple requests concurrently using asyncio"""
async def get_response(prompt: str) -> str:
    response = await acompletion(
        model="claude-sonnet-4-6",
        messages=[{"role": "user", "content": prompt}]
    )
    return response.choices[0].message.content

prompts = [
    "What is 2 + 2?",
    "What is the capital of Japan?",
    "Who wrote Romeo and Juliet?"
]

# Execute all requests concurrently
tasks = [get_response(p) for p in prompts]
results = await asyncio.gather(*tasks)

for prompt, result in zip(prompts, results):
    print(f"Q: {prompt}")
    print(f"A: {result}\n")
     

In [ ]:
import asyncio
from litellm import acompletion

"""Extend the async example to include rate limiting and automatic retry."""
# Limit to 10 concurrent requests
semaphore = asyncio.Semaphore(10)

async def call_llm(prompt: str) -> str:
    """LLM call with rate limiting and automatic retry."""
    async with semaphore:
        response = await acompletion(
            model="claude-sonnet-4-6",
            messages=[{"role": "user", "content": prompt}],
            num_retries=3  # Automatic retry with exponential backoff
        )
        return response.choices[0].message.content

# Even with 100 concurrent tasks, only 10 API calls run at a time
prompts = [f"What is {i} + {i}?" for i in range(100)]
tasks = [call_llm(p) for p in prompts]
results = await asyncio.gather(*tasks, return_exceptions=True)

successes = sum(1 for r in results if not isinstance(r, Exception))
print(f"Successful: {successes}/{len(results)}")

In [ ]:
from datasets import load_dataset

level1_problems = load_dataset("gaia-benchmark/GAIA", "2023_level1", split="validation")
print(f"Number of Level 1 problems: {len(level1_problems)}")


In [ ]:
from pydantic import BaseModel

class GaiaOutput(BaseModel):
    is_solvable: bool
    unsolvable_reason: str = ""
    final_answer: str = ""

PROVIDER_SEMAPHORES = {
    "openai" : asyncio.Semaphore(30),
    "anthropic": asyncio.Semaphore(10),
}

def get_provider(model: str) -> str:
    """Extract provider name from model string"""
    return "anthropic" if model.startswith("anthropic/") else "openai"

async def solve_problem(model: str, question: str) -> GaiaOutput:
    """Solve a single problem and return a structured output"""
    provider = get_provider(model)

    async with PROVIDER_SEMAPHORES[provider]:
         response = await acompletion(
            model=model,
            messages=[
                 {"role": "system", "content": gaia_promt},
                 {"role": "user", "content": question},
            ],
            response_format=GaiaOutput
            num_retries=2  # Automatic retry with exponential backoff
            )
         finish_reason = response.choices[0].finish_reason
         content = response.choices[0].message.content

         if finish_reason == "refusal" or content is None:
             return GaiaOutput(
                 is_solvable=False,
                 unsolvable_reason=f"Model refused to answer (finish_reason: {finish_reason})",
                 final_answer=""
             )
         return GaiaOutput.model_validate_json(content)

#check if it matches the exact answer
def is_correct(prediction: str | None, answer: str) -> bool:
    """Check exact match between prediction and answer"""
    if prediction is None:
        return False
    #compare prediction for exact match
    return prediction.strip().lower() == answer.strip().lower()

async def evaluate_gaia_single(problem: dict, model: str) -> dict:
    """Evaluate a single problem-model pair and return result."""
    try:
        output = await solve_problem(model, problem["Question"])
        return {
            "task_id": problem["task_id"],
            "model": model,
            "correct": is_correct(output.final_answer, problem["Final answer"]),
            "is_solvable": output.is_solvable,
            "prediction": output.final_answer,
            "answer": problem["Final answer"],
            "unsolvable_reason": output.unsolvable_reason,
        }
    except Exception as e:
        return {
            "task_id": problem["task_id"],
            "model": model,
            "correct": False,
            "is_solvable": None,
            "prediction": None,
            "answer": problem["Final answer"],
            "error": str(e),
        }

#full-batch running all problems and models concurrently grouping results by model
async def run_experiment(
    problems: list[dict],
    models: list[str],
    ) -> dict[str, list]:
    """Evaluate all models on all problems."""
    tasks = [
        evaluate_gaia_single(problem, model)
        for problem in problems
        for model in models
    ]
    all_results = await tqdm_asyncio.gather(*tasks)

    # Group results by model
    results = {model: [] for model in models}
    for result in all_results:
        results[result["model"]].append(result)
    return results


In [ ]:
from datasets import load_dataset
from scratch_agents.eval.gaia import run_experiment

"""Example of running an experiment with multiple models and collecting results."""

MODELS = [
    "gpt-5.5",
    "gpt-5.4-mini",
    "anthropic/claude-sonnet-4-6",
    "anthropic/claude-haiku-4-5"
]

subset = level1_problems.select(range(20))
results = await run_experiment(subset, MODELS)

# Per-model accuracy
for model, model_results in results.items():
    correct = sum(1 for r in model_results if r["correct"])
    solvable = sum(1 for r in model_results if r.get("is_solvable"))
    total = len(model_results)
    print(f"{model}: {correct}/{total} correct, {solvable}/{total} judged solvable")